# EDA: Flow-Based Gentrification in London (Option C)
**Dissertation Meeting 1 — Prepared Materials**

### Methodology: Option C — Fixed Baseline + IMD-Change Validation
- **Cascading Flow Index**: Uses IMD 2010 (fixed baseline) to assign wealth deciles,
  then measures cascade flows for both 2011 and 2021 census periods.
- **Validation**: Independently compute IMD change (2010 → 2019) per MSOA.
  If the cascade index correctly identifies gentrifying areas, those MSOAs
  should also show falling IMD scores (less deprived over time).

### Geography
- All analysis harmonised to **2011 MSOA codes**.
- 2021 census O-D data (which uses 2021 MSOA codes) is converted using the
  ONS MSOA 2011-to-2021 lookup. MSOAs that were split/merged/redesigned
  are excluded (~2.5% nationally, fewer in London).

---

### Contents
1. Data loading & overview
2. Geography harmonisation (2021 → 2011 MSOA codes)
3. London filter & IMD aggregation to MSOA (both 2010 and 2019)
4. Wealth decile assignment (fixed 2010 baseline)
5. IMD change map (validation layer)
6. O-D flow heatmaps (2011 & 2021)
7. Cascading displacement signal
8. Per-MSOA cascade features
9. Borough-level summary
10. Key findings for supervisor discussion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT = here()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

---
## 1. Load All Datasets

**Files you need in your `data/` folder:**

| File | Description | Download |
|------|-------------|----------|
| `imd_2010.csv` | IMD 2010 scores at LSOA level | [GOV.UK — English Indices of Deprivation 2010](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2010) (File 1 xlsx → save as csv) |
| `imd_2019.csv` | IMD 2019 scores at LSOA level | You already have this as `imd.csv` |
| `census_od_2011.csv` | 2011 Census migration O-D (MSOA) | [Nomis — 2011 Census Origin-Destination](https://www.nomisweb.co.uk/census/2011/origin_destination) → Table MM01 at MSOA level |
| `ODMG01EW_MSOA.csv` | 2021 Census migration O-D (MSOA) | You already have this |
| `NSPCL_NOV22_UK_LU.csv` | Postcode lookup (2011 geographies) | You already have this |
| `msoa_2011_to_2021_lookup.csv` | MSOA 2011↔2021 code mapping | [Open Geography Portal](https://geoportal.statistics.gov.uk/datasets/ons::msoa-2011-to-msoa-2021-to-local-authority-district-2022-best-fit-lookup-for-ew-v2/about) |

In [ ]:
# ---- File paths ----
imd_2010_path     = DATA_DIR / 'imd_2010.csv'          # IMD 2010 (LSOA-level)
imd_2019_path     = DATA_DIR / 'imd.csv'               # IMD 2019 (LSOA-level) — your existing file
census_od_2011_path = DATA_DIR / 'census_od_2011.csv'   # 2011 Census MM01 (MSOA)
census_od_2021_path = DATA_DIR / 'ODMG01EW_MSOA.csv'    # 2021 Census ODMG01 (MSOA)
lookup_path       = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'  # Postcode → LSOA/MSOA (2011)
msoa_lookup_path  = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'  # MSOA 2011 ↔ 2021

In [ ]:
# Load IMD datasets
imd_2019 = pd.read_csv(imd_2019_path)
imd_2019.columns = imd_2019.columns.str.strip()

# ---- IMD 2010 ----
# NOTE: The IMD 2010 file from GOV.UK is an Excel file. You may need to:
#   1. Open the .xlsx, find the sheet with LSOA-level IMD scores
#   2. Save it as CSV, or adjust the read call below:
#
# imd_2010 = pd.read_excel(DATA_DIR / 'imd_2010.xlsx', sheet_name='IMD 2010')
#
# Common column names in IMD 2010: 'LSOA CODE', 'IMD SCORE'
# Adjust the column names in the code below to match your file.

imd_2010 = pd.read_csv(imd_2010_path)
imd_2010.columns = imd_2010.columns.str.strip()

print('=== IMD 2010 columns ===')
print(imd_2010.columns.tolist())
print(f'Shape: {imd_2010.shape}')
display(imd_2010.head(3))

print('\n=== IMD 2019 columns ===')
print(imd_2019.columns.tolist())
print(f'Shape: {imd_2019.shape}')
display(imd_2019.head(3))

In [ ]:
# Load census O-D data
census_od_2021 = pd.read_csv(census_od_2021_path)

# ---- 2011 Census O-D ----
# NOTE: The 2011 MM01 table may have different column names.
# Typical columns: origin MSOA code, destination MSOA code, count.
# After loading, inspect columns and rename to standardise.

census_od_2011 = pd.read_csv(census_od_2011_path)

print('=== 2021 Census O-D columns ===')
print(census_od_2021.columns.tolist())
print(f'Shape: {census_od_2021.shape}')

print('\n=== 2011 Census O-D columns ===')
print(census_od_2011.columns.tolist())
print(f'Shape: {census_od_2011.shape}')

In [ ]:
# Load lookup tables
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)

msoa_11_21 = pd.read_csv(msoa_lookup_path)
print('=== MSOA 2011-2021 Lookup columns ===')
print(msoa_11_21.columns.tolist())
display(msoa_11_21.head(3))

---
## 2. Geography Harmonisation: Convert 2021 MSOA Codes → 2011

The 2021 census O-D data uses 2021 MSOA codes, but our IMD data and
postcode lookup use 2011 MSOA codes. Most MSOAs are unchanged, but ~184
were split, merged, or redesigned.

**Strategy**: Keep only MSOAs with a clean 1:1 mapping (marked 'U' for
unchanged in the lookup). Drop splits/merges/irregular for now.

In [ ]:
# ---- Inspect the MSOA lookup ----
# The lookup should have columns like:
#   MSOA11CD, MSOA21CD, CHGIND (change indicator: U/S/M/X)
#
# Adjust column names below to match your file.
# Common variants: 'msoa11cd'/'MSOA11CD', 'msoa21cd'/'MSOA21CD', 'chgind'/'CHGIND'

# Standardise column names to lowercase
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()
print(msoa_11_21.columns.tolist())

# Check change indicator distribution
print('\nChange indicator counts:')
print(msoa_11_21['chgind'].value_counts())

In [ ]:
# Keep only unchanged MSOAs (1:1 mapping between 2011 and 2021)
unchanged = msoa_11_21[msoa_11_21['chgind'] == 'U'].copy()

# Create a mapping dictionary: 2021 code → 2011 code
# For unchanged MSOAs, the codes are often identical, but let's be safe
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))

print(f'Total MSOAs in lookup: {len(msoa_11_21)}')
print(f'Unchanged (usable): {len(unchanged)}')
print(f'Dropped (split/merged/irregular): {len(msoa_11_21) - len(unchanged)}')

# Check how many London MSOAs are affected
# (We'll verify this after filtering to London)

In [ ]:
# ---- Convert 2021 census O-D data to 2011 MSOA codes ----

# Column names in your 2021 O-D file:
ORIGIN_COL_2021 = 'Migrant MSOA one year ago code'     # origin (2021 MSOA code)
DEST_COL_2021   = 'Middle layer Super Output Areas code' # destination (2021 MSOA code)

# Map to 2011 codes
census_od_2021['origin_msoa11'] = census_od_2021[ORIGIN_COL_2021].map(msoa21_to_11)
census_od_2021['dest_msoa11']   = census_od_2021[DEST_COL_2021].map(msoa21_to_11)

# Check coverage
n_total = len(census_od_2021)
n_mapped = census_od_2021[['origin_msoa11', 'dest_msoa11']].notna().all(axis=1).sum()
print(f'2021 O-D records: {n_total:,}')
print(f'Both origin & dest mapped to 2011 codes: {n_mapped:,} ({n_mapped/n_total*100:.1f}%)')
print(f'Dropped (split/merged MSOAs): {n_total - n_mapped:,}')

---
## 3. London Filter & IMD Aggregation to MSOA

In [ ]:
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

# Get unique LSOA → MSOA (2011) mapping for London
london_lookup = (
    lookup[lookup['ladnm'].isin(london_boroughs)]
    [['lsoa11cd', 'msoa11cd', 'ladnm']]
    .drop_duplicates()
)

print(f'London LSOAs in lookup: {london_lookup["lsoa11cd"].nunique()}')
print(f'London MSOAs in lookup: {london_lookup["msoa11cd"].nunique()}')
print(f'Boroughs: {london_lookup["ladnm"].nunique()}')

In [ ]:
# ---- Aggregate IMD 2010 to MSOA level ----
#
# IMPORTANT: Adjust column names below to match your IMD 2010 file.
# Common names: 'LSOA CODE' or 'LSOA code (2011)', 'IMD SCORE' or 'IMD Score'

IMD_2010_LSOA_COL  = 'LSOA CODE'   # <-- adjust to your file's column name
IMD_2010_SCORE_COL = 'IMD SCORE'    # <-- adjust to your file's column name

imd_2010_london = pd.merge(
    imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]],
    london_lookup,
    left_on=IMD_2010_LSOA_COL,
    right_on='lsoa11cd'
)

msoa_imd_2010 = (
    imd_2010_london
    .groupby(['msoa11cd', 'ladnm'])[IMD_2010_SCORE_COL]
    .mean()
    .reset_index()
    .rename(columns={IMD_2010_SCORE_COL: 'IMD_2010'})
)

print(f'London MSOAs with IMD 2010: {len(msoa_imd_2010)}')

In [ ]:
# ---- Aggregate IMD 2019 to MSOA level ----

IMD_2019_LSOA_COL  = 'LSOA code (2011)'
IMD_2019_SCORE_COL = 'Index of Multiple Deprivation (IMD) Score'

imd_2019_london = pd.merge(
    imd_2019[[IMD_2019_LSOA_COL, IMD_2019_SCORE_COL]],
    london_lookup,
    left_on=IMD_2019_LSOA_COL,
    right_on='lsoa11cd'
)

msoa_imd_2019 = (
    imd_2019_london
    .groupby('msoa11cd')[IMD_2019_SCORE_COL]
    .mean()
    .reset_index()
    .rename(columns={IMD_2019_SCORE_COL: 'IMD_2019'})
)

print(f'London MSOAs with IMD 2019: {len(msoa_imd_2019)}')

In [ ]:
# ---- Combine IMD 2010 + 2019 per MSOA ----

msoa_wealth = pd.merge(msoa_imd_2010, msoa_imd_2019, on='msoa11cd', how='inner')

# IMD change (negative = area became less deprived = potential gentrification)
msoa_wealth['IMD_Change'] = msoa_wealth['IMD_2019'] - msoa_wealth['IMD_2010']

print(f'London MSOAs with both IMD years: {len(msoa_wealth)}')
msoa_wealth.head()

---
## 4. Wealth Deciles (Fixed 2010 Baseline)

We assign deciles using **IMD 2010 only**, so the classification is
identical for the 2011 and 2021 flow analysis.

In [ ]:
# Decile based on IMD 2010 (1 = most deprived, 10 = least deprived / wealthiest)
msoa_wealth['Wealth_Decile'] = pd.qcut(
    msoa_wealth['IMD_2010'], 10, labels=False
) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']

wealth_dict = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()

print('Decile distribution:')
print(msoa_wealth['Wealth_Decile'].value_counts().sort_index())

---
## 5. IMD Change Analysis (Validation Layer)

This is the independent check: which MSOAs got less deprived between
2010 and 2019? Later we will compare this with our cascade index.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 5a. Distribution of IMD change
axes[0].hist(msoa_wealth['IMD_Change'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('IMD Change (2019 − 2010)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('IMD Score Change across London MSOAs\n(negative = less deprived / potential gentrification)')

# 5b. IMD change by baseline decile
decile_change = msoa_wealth.groupby('Wealth_Decile')['IMD_Change'].mean()
colors = ['coral' if v < 0 else 'steelblue' for v in decile_change.values]
axes[1].bar(decile_change.index, decile_change.values, color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Wealth Decile (2010 baseline)')
axes[1].set_ylabel('Mean IMD Change')
axes[1].set_title('Average IMD Change by 2010 Wealth Decile')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_imd_change.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Origin–Destination Flow Analysis

We process both census years using the same fixed wealth deciles.

In [ ]:
def prepare_od_data(od_df, origin_col, dest_col, year_label):
    """
    Clean O-D data and assign wealth deciles.
    Returns London-only flows with decile info.
    """
    df = od_df.copy()
    
    # Remove non-migrant codes
    df = df[df[origin_col].astype(str) != '-8'].copy()
    
    # Map deciles
    df['Origin_Decile'] = df[origin_col].map(wealth_dict)
    df['Dest_Decile']   = df[dest_col].map(wealth_dict)
    
    # Keep only London flows (both endpoints have a decile)
    df = df.dropna(subset=['Origin_Decile', 'Dest_Decile']).copy()
    df['Origin_Decile'] = df['Origin_Decile'].astype(int)
    df['Dest_Decile']   = df['Dest_Decile'].astype(int)
    
    # Detect count column
    count_cols = [c for c in df.columns if 'observation' in c.lower() or 'count' in c.lower()]
    if count_cols:
        df['flow_count'] = df[count_cols[0]]
    else:
        df['flow_count'] = 1
    
    df['Decile_Shift'] = df['Dest_Decile'] - df['Origin_Decile']
    
    print(f'--- {year_label} ---')
    print(f'  London flows: {len(df):,}')
    print(f'  Total migrants: {df["flow_count"].sum():,.0f}')
    
    return df

In [ ]:
# ---- Process 2021 data (using harmonised 2011 codes) ----
flow_2021 = prepare_od_data(
    census_od_2021,
    origin_col='origin_msoa11',   # harmonised column from step 2
    dest_col='dest_msoa11',
    year_label='2021'
)

# ---- Process 2011 data (already uses 2011 codes) ----
# NOTE: Adjust column names to match your 2011 MM01 file.
# Common names: 'Area of usual residence' / 'Area of residence one year ago'
# Inspect census_od_2011.columns and set the correct names below:

ORIGIN_COL_2011 = 'TODO_ADJUST'  # <-- column for "residence 1 year ago" MSOA code
DEST_COL_2011   = 'TODO_ADJUST'  # <-- column for "current residence" MSOA code

flow_2011 = prepare_od_data(
    census_od_2011,
    origin_col=ORIGIN_COL_2011,
    dest_col=DEST_COL_2011,
    year_label='2011'
)

In [ ]:
# ---- Flow heatmaps: 2011 vs 2021 side by side ----

def make_flow_matrix(flow_df):
    return flow_df.pivot_table(
        index='Origin_Decile', columns='Dest_Decile',
        values='flow_count', aggfunc='sum', fill_value=0
    )

mat_2011 = make_flow_matrix(flow_2011)
mat_2021 = make_flow_matrix(flow_2021)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, mat, title in [(axes[0], mat_2011, '2011 Census'),
                        (axes[1], mat_2021, '2021 Census')]:
    sns.heatmap(
        mat, annot=True, fmt=',.0f', cmap='YlOrRd',
        linewidths=0.5, linecolor='white',
        cbar_kws={'label': 'Migrants'},
        ax=ax
    )
    ax.set_xlabel('Destination Wealth Decile')
    ax.set_ylabel('Origin Wealth Decile')
    ax.set_title(f'Migration Flows: {title}\n(Fixed IMD 2010 Deciles)')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_od_heatmap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Cascading Displacement Signal: 2011 vs 2021

In [ ]:
def flow_direction_summary(flow_df, year_label):
    """Classify flows as upward/downward/lateral and print summary."""
    total = flow_df['flow_count'].sum()
    
    upward   = flow_df[flow_df['Decile_Shift'] > 0]['flow_count'].sum()
    downward = flow_df[flow_df['Decile_Shift'] < 0]['flow_count'].sum()
    lateral  = flow_df[flow_df['Decile_Shift'] == 0]['flow_count'].sum()
    
    print(f'\n=== {year_label} Flow Directions ===')
    print(f'  Upward (to wealthier):      {upward:>10,.0f}  ({upward/total*100:.1f}%)')
    print(f'  Downward (to more deprived): {downward:>10,.0f}  ({downward/total*100:.1f}%)')
    print(f'  Lateral (same decile):       {lateral:>10,.0f}  ({lateral/total*100:.1f}%)')
    print(f'  Total:                       {total:>10,.0f}')
    
    return {'upward': upward, 'downward': downward, 'lateral': lateral, 'total': total}

summary_2011 = flow_direction_summary(flow_2011, '2011')
summary_2021 = flow_direction_summary(flow_2021, '2021')

In [ ]:
# ---- Decile shift distribution: 2011 vs 2021 ----

shift_2011 = flow_2011.groupby('Decile_Shift')['flow_count'].sum()
shift_2021 = flow_2021.groupby('Decile_Shift')['flow_count'].sum()

# Normalise to percentages for fair comparison
shift_2011_pct = shift_2011 / shift_2011.sum() * 100
shift_2021_pct = shift_2021 / shift_2021.sum() * 100

all_shifts = sorted(set(shift_2011_pct.index) | set(shift_2021_pct.index))

fig, ax = plt.subplots(figsize=(11, 5))
width = 0.35
x = np.arange(len(all_shifts))

bars_2011 = [shift_2011_pct.get(s, 0) for s in all_shifts]
bars_2021 = [shift_2021_pct.get(s, 0) for s in all_shifts]

ax.bar(x - width/2, bars_2011, width, label='2011', color='steelblue', edgecolor='white')
ax.bar(x + width/2, bars_2021, width, label='2021', color='coral', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(all_shifts)
ax.set_xlabel('Decile Shift (negative = moved to more deprived area)')
ax.set_ylabel('% of all migrants')
ax.set_title('Decile Shift Distribution: 2011 vs 2021 (Fixed IMD 2010 Baseline)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_shift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Per-MSOA Cascade Features (Both Years)

In [ ]:
def compute_cascade_features(flow_df, origin_col, dest_col, suffix):
    """
    For each MSOA, compute cascade metrics.
    origin_col / dest_col: the MSOA code columns in flow_df (should be 2011 codes).
    suffix: e.g. '_2011' or '_2021'
    """
    # Inflow from wealthier: people arriving whose origin decile > dest decile
    inflow_w = (
        flow_df[flow_df['Origin_Decile'] > flow_df['Dest_Decile']]
        .groupby(dest_col)['flow_count'].sum()
        .rename(f'Inflow_Wealthier{suffix}')
    )
    
    # Outflow to poorer: people leaving whose dest decile < origin decile
    outflow_p = (
        flow_df[flow_df['Dest_Decile'] < flow_df['Origin_Decile']]
        .groupby(origin_col)['flow_count'].sum()
        .rename(f'Outflow_Poorer{suffix}')
    )
    
    # Total inflow & outflow
    total_in = flow_df.groupby(dest_col)['flow_count'].sum().rename(f'Total_Inflow{suffix}')
    total_out = flow_df.groupby(origin_col)['flow_count'].sum().rename(f'Total_Outflow{suffix}')
    
    return inflow_w, outflow_p, total_in, total_out

In [ ]:
# Start with base MSOA table
msoa_analysis = msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile', 'IMD_2010', 'IMD_2019', 'IMD_Change']].copy()

# ---- 2021 cascade features ----
iw21, op21, ti21, to21 = compute_cascade_features(
    flow_2021, 'origin_msoa11', 'dest_msoa11', '_2021'
)
for series in [iw21, op21, ti21, to21]:
    msoa_analysis = msoa_analysis.merge(series, left_on='msoa11cd', right_index=True, how='left')

# ---- 2011 cascade features ----
# Adjust column names to match your 2011 data
iw11, op11, ti11, to11 = compute_cascade_features(
    flow_2011, ORIGIN_COL_2011, DEST_COL_2011, '_2011'
)
for series in [iw11, op11, ti11, to11]:
    msoa_analysis = msoa_analysis.merge(series, left_on='msoa11cd', right_index=True, how='left')

msoa_analysis = msoa_analysis.fillna(0)

# ---- Derived features ----
for suffix in ['_2011', '_2021']:
    iw = f'Inflow_Wealthier{suffix}'
    op = f'Outflow_Poorer{suffix}'
    ti = f'Total_Inflow{suffix}'
    
    msoa_analysis[f'Net_Cascade{suffix}'] = msoa_analysis[iw] - msoa_analysis[op]
    msoa_analysis[f'Cascade_Ratio{suffix}'] = np.where(
        msoa_analysis[op] > 0,
        msoa_analysis[iw] / msoa_analysis[op],
        np.nan
    )
    msoa_analysis[f'Pct_Inflow_Wealthier{suffix}'] = (
        msoa_analysis[iw] / msoa_analysis[ti].replace(0, np.nan) * 100
    )

# Change in cascade between periods
msoa_analysis['Cascade_Change'] = msoa_analysis['Net_Cascade_2021'] - msoa_analysis['Net_Cascade_2011']

msoa_analysis.head()

---
## 9. Validation: Does Cascade Index Predict IMD Change?

The key test for Option C: MSOAs with high cascade pressure (affluent
in-migration displacing residents) should independently show falling
IMD scores between 2010 and 2019.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 9a. Net Cascade (2011) vs IMD Change
ax = axes[0]
sc = ax.scatter(
    msoa_analysis['Net_Cascade_2011'],
    msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Net Cascade (2011 flows)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('Does 2011 Cascade Predict Future IMD Change?')

# 9b. Net Cascade (2021) vs IMD Change
ax = axes[1]
sc = ax.scatter(
    msoa_analysis['Net_Cascade_2021'],
    msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Net Cascade (2021 flows)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('Does 2021 Cascade Correlate with IMD Change?')
plt.colorbar(sc, label='Wealth Decile (2010)', ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig4_cascade_vs_imd_change.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation statistics
from scipy import stats

for col, label in [('Net_Cascade_2011', '2011 Cascade'), ('Net_Cascade_2021', '2021 Cascade')]:
    valid = msoa_analysis[[col, 'IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Change'])
    print(f'{label} vs IMD Change: r = {r:.3f}, p = {p:.4f}')

---
## 10. Borough-Level Summary

In [ ]:
borough_summary = msoa_analysis.groupby('ladnm').agg(
    Num_MSOAs=('msoa11cd', 'count'),
    Avg_IMD_2010=('IMD_2010', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
    Net_Cascade_2011=('Net_Cascade_2011', 'sum'),
    Net_Cascade_2021=('Net_Cascade_2021', 'sum'),
    Cascade_Change=('Cascade_Change', 'sum'),
).round(2).sort_values('Cascade_Change', ascending=False)

display(borough_summary)

In [ ]:
# Borough comparison: cascade 2011 vs 2021
fig, ax = plt.subplots(figsize=(10, 9))

y = range(len(borough_summary))
ax.barh([i + 0.15 for i in y], borough_summary['Net_Cascade_2011'],
        height=0.3, label='2011', color='steelblue', edgecolor='white')
ax.barh([i - 0.15 for i in y], borough_summary['Net_Cascade_2021'],
        height=0.3, label='2021', color='coral', edgecolor='white')
ax.set_yticks(list(y))
ax.set_yticklabels(borough_summary.index)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Net Cascade')
ax.set_title('Net Cascade by Borough: 2011 vs 2021')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig5_borough_cascade_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Export

In [ ]:
msoa_analysis.to_csv(OUTPUT_DIR / 'msoa_cascade_features.csv', index=False)
borough_summary.to_csv(OUTPUT_DIR / 'borough_summary.csv')
mat_2011.to_csv(OUTPUT_DIR / 'flow_matrix_2011.csv')
mat_2021.to_csv(OUTPUT_DIR / 'flow_matrix_2021.csv')

print('Done. Exported to outputs/ folder.')

---
## Supervisor Discussion Points

1. **Option C methodology**: Fixed 2010 IMD baseline for flow classification,
   with IMD change (2010→2019) as independent validation. Does this approach
   make sense, or should we also try period-matched IMD as a sensitivity check?

2. **Geography harmonisation**: We dropped ~184 MSOAs nationally (splits/merges)
   to maintain consistent 2011 MSOA codes. How many London MSOAs are affected?
   Should we attempt population-weighted reaggregation for changed MSOAs?

3. **Cascade Index formalisation**: Current metric is
   `Net_Cascade = Inflow_from_wealthier − Outflow_to_poorer`. Should we
   normalise by total flow volume? Weight by magnitude of decile shift?

4. **COVID caveat**: The 2021 Census was conducted on 21 March 2021, during
   COVID restrictions. Migration patterns in the preceding year may be
   atypical. How do we address this in the dissertation?

5. **Next steps**: Alluvial plots, spatial mapping with geopandas, and
   potentially ODMG04EW (migration by NS-SEC) for socio-economic class
   instead of area-based IMD proxy.